# Session 02 · Working with MongoDB Databases

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - databases and collections

In [2]:
from mongo_conn import get_client
client = get_client()
db = client["campus_admin"]
for c in db.list_collection_names():
    db[c].drop()
db.notices.insert_one({"text": "Eid holidays from Monday"})
print("databases:", client.list_database_names())
print("collections:", db.list_collection_names())
db.notices.rename("announcements")
print("after rename:", db.list_collection_names())

databases: ['campus_admin']
collections: ['notices']
after rename: ['announcements']


## Example 2 - ordered vs unordered inserts

In [3]:
from pymongo.errors import BulkWriteError
for ordered in (True, False):
    db.t.drop()
    try:
        db.t.insert_many([{"_id": 1}, {"_id": 1}, {"_id": 2}], ordered=ordered)
    except BulkWriteError as e:
        print(f"ordered={ordered}: inserted {e.details['nInserted']} of 3")

ordered=True: inserted 1 of 3
ordered=False: inserted 2 of 3


## Example 3 - bulk writes

In [4]:
from pymongo import DeleteOne, InsertOne
res = db.batches.bulk_write([InsertOne({"_id": "WEB-01"}), InsertOne({"_id": "NET-01"}), DeleteOne({"_id": "NET-01"})])
print("inserted", res.inserted_count, "| deleted", res.deleted_count)

inserted 2 | deleted 1


## Example 4 - embedding

In [5]:
db.batches.insert_one({"_id": "DS-01", "faculty": {"name": "Aseef Ahmed", "subject": "Python"},
                       "slots": [{"day": "Mon", "time": "18:00"}, {"day": "Wed", "time": "18:00"}]})
print(db.batches.find_one({"_id": "DS-01"}))
print("faculty only:", db.batches.find_one({"_id": "DS-01"}, {"faculty.name": 1, "_id": 0}))

{'_id': 'DS-01', 'faculty': {'name': 'Aseef Ahmed', 'subject': 'Python'}, 'slots': [{'day': 'Mon', 'time': '18:00'}, {'day': 'Wed', 'time': '18:00'}]}
faculty only: {'faculty': {'name': 'Aseef Ahmed'}}


## Example 5 - referencing and many-to-many

In [6]:
db.students.insert_many([{"student_id": 1, "name": "Ali Raza"}, {"student_id": 2, "name": "Sana Khan"}])
db.payments.insert_many([{"student_id": 2, "amount": 15000, "month": "2026-09"},
                         {"student_id": 2, "amount": 15000, "month": "2026-10"}])
db.students.update_one({"student_id": 2}, {"$set": {"course_ids": ["PY101", "DB201"]}})
db.students.update_one({"student_id": 1}, {"$set": {"course_ids": ["PY101"]}})
sana = db.students.find_one({"name": "Sana Khan"})
print("Sana paid:", sum(p["amount"] for p in db.payments.find({"student_id": sana["student_id"]})))
print("taking DB201:", [s["name"] for s in db.students.find({"course_ids": "DB201"})])

Sana paid: 30000
taking DB201: ['Sana Khan']


## Example 6 - validating documents

In [7]:
from schema_check import STUDENT_SCHEMA, check
print(check({"name": "Sana Khan", "batch": "DS-01", "marks": {"python": 90}}, STUDENT_SCHEMA))
print(check({"name": "Al", "batch": "XX-9", "marks": {"python": 120}}, STUDENT_SCHEMA))

[]
['name too short', 'batch not allowed', 'marks.python out of range']
